# Multi-Head Attention

## Start with the problem: one similarity space can be restrictive

A single attention operation compares queries and keys in one learned feature space. Multi-head attention creates several lower-dimensional projections, computes attention independently in each, and combines their outputs. The heads can learn different interactions, though specialization is not guaranteed.

This lesson traces the projections, dimensions, concatenation, and output transform.

## Follow the parallel heads into one output

```text
             -> head 1 --+
input Q,K,V -> -> head 2 --+-> concatenate -> output projection
             -> head H --+
```

Imagine two positions with value vectors $v_1=[1,0]$ and $v_2=[0,2]$. If head 1 places nearly all its weight on position 1, its read is about $[1,0]$; if head 2 favors position 2, its read is about $[0,2]$. Concatenation produces approximately $[1,0,0,2]$, which the learned output projection can mix.

With batch size $B=1$, sequence length $L=3$, model width $8$, and two heads of width $4$, each head's score matrix has shape $1\times3\times3$. Stacking the heads gives $1\times2\times3\times3$ before the value reads.

## Compute several reads in parallel

For head $j$, learned matrices produce $Q_j=QW_j^Q$, $K_j=KW_j^K$, and $V_j=VW_j^V$. Then:

$$head_j=\operatorname{softmax}(Q_jK_j^T/\sqrt{d_k})V_j,$$

$$\operatorname{MHA}(Q,K,V)=\operatorname{Concat}(head_1,\ldots,head_H)W^O.$$

Each head has its own projection parameters. In common Transformer configurations, $H d_k$ is chosen to relate to the model width, but implementations can use other dimensions.

## Keep the tensor dimensions straight

For batch size $B$, query length $L_q$, key length $L_k$, and head width $d_k$, each head forms score tensors shaped approximately $B\times H\times L_q\times L_k$. The value read has shape $B\times H\times L_q\times d_v$. Concatenating heads restores the combined channel dimension before the output projection.

In self-attention, queries, keys, and values come from the same sequence representation. In cross-attention, queries come from one stream and keys/values from another. Padding and causal masks must broadcast over the intended batch and head axes.

## Understand the tradeoff

More heads create more learned subspaces, but at a fixed model width they often make each head narrower. Head count does not multiply useful independent reasoning automatically. The attention score matrix still scales with sequence-pair count, and mask or shape mistakes can silently alter information flow.

**Takeaway:** multi-head attention is parallel attention with separate learned projections, followed by concatenation and mixing. Verify head dimensions and masks explicitly; do not infer semantic roles from head weights alone.